<h1 style="text-align: center; font-weight: bold;">STACKING ENSEMBLE MODEL</h1>

## Loading Dataset

In [16]:
# Import libraries
import pandas as pd
import numpy as np
from mlxtend.classifier import StackingClassifier
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.linear_model import LogisticRegression, SGDClassifier, PassiveAggressiveClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix
from tabulate import tabulate

In [2]:
# Load proccessed dataset
data = pd.read_csv("../data/processed_train.csv")
# Load test dataset
test_data = pd.read_csv("../data/processed_test.csv")

# Drop id column
data = data.drop(columns=['id'])
# Keep test id column separately
test_ids = test_data['id']
# Drop test id column
test_data = test_data.drop(columns=['id'])

data.head()

,Name,Gender,Age,City,Working Professional or Student,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Overall Stress Level,Depression
0,Aaradhya,0,49.0,Ludhiana,1,Chef,0.0,5.0,0.0,0.0,2.0,9.0,Healthy,BHM,0,1.0,2.0,0,5.0,0
1,Vivan,1,26.0,Varanasi,1,Teacher,0.0,4.0,0.0,0.0,3.0,4.0,Unhealthy,LLB,1,7.0,3.0,0,4.0,1
2,Yuvraj,1,33.0,Visakhapatnam,0,Not Applicable,5.0,0.0,5.5,2.0,0.0,5.5,Healthy,B.Pharm,1,3.0,1.0,0,4.0,1
3,Yuvraj,1,22.0,Mumbai,1,Teacher,0.0,5.0,0.0,0.0,1.0,4.0,Moderate,BBA,1,10.0,1.0,1,5.0,1
4,Rhea,0,30.0,Kanpur,1,Business Analyst,0.0,1.0,0.0,0.0,1.0,5.5,Unhealthy,BBA,1,9.0,4.0,1,4.0,0


## Define Constants

In [3]:
LABEL_COL = 'Depression'
NUMERIC_COLS = ['Age', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction',
                'Sleep Duration', 'Work/Study Hours', 'Financial Stress', 'Overall Stress Level']
CATEGORICAL_COLS = ['Name', 'City', 'Profession', 'Dietary Habits', 'Degree', ]

# Meta model for stacking ensemble
META_MODEL = LogisticRegression(solver='liblinear',
                                class_weight='balanced',
                                random_state=42)

NUM_FOLD = 10

## Data Preprocessing

In [4]:
# Separate features and labels
X = data.drop(columns=[LABEL_COL])
y = data[LABEL_COL]

# Normalize numeric features
scaler = StandardScaler()
X[NUMERIC_COLS] = scaler.fit_transform(X[NUMERIC_COLS])
test_data[NUMERIC_COLS] = scaler.transform(test_data[NUMERIC_COLS])

# Encode categorical features
encoder = TargetEncoder()
X[CATEGORICAL_COLS] = encoder.fit_transform(X[CATEGORICAL_COLS], y)
test_data[CATEGORICAL_COLS] = encoder.transform(test_data[CATEGORICAL_COLS])

## Tuning Hyperparameters for Base Models

### `DecisionTreeClassifier`

In [5]:
# Tune DecisionTreeClassifier with GridSearchCV

# Define parameter grid for DecisionTreeClassifier
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 10],
    'class_weight': [None, 'balanced']
}

dt_base = DecisionTreeClassifier(random_state=42)
gs = GridSearchCV(dt_base, param_grid, cv=NUM_FOLD, scoring='f1', n_jobs=-1, verbose=2)
gs.fit(X, y)

print("Best params:", gs.best_params_)
print("Best CV f1 score:", gs.best_score_)

# Update dt_model with the best estimator from GridSearchCV
dt_model = gs.best_estimator_

Fitting 10 folds for each of 256 candidates, totalling 2560 fits
Best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 10, 'min_samples_split': 2}
Best CV f1 score: 0.8084631470715665


### `LinearSVC`

In [6]:
# Tune LinearSVC with GridSearchCV

# Define parameter grid for LinearSVC
param_grid_svc = {
    'C': [0.01, 0.1, 1, 10, 100],
    'class_weight': [None, 'balanced'],
    'max_iter': [100, 500, 1000]
}

svc_base = LinearSVC(random_state=42)
gs_svc = GridSearchCV(svc_base, param_grid_svc, cv=NUM_FOLD, scoring='f1', n_jobs=-1, verbose=2)
gs_svc.fit(X, y)

print("Best params for LinearSVC:", gs_svc.best_params_)
print("Best CV f1 score for LinearSVC:", gs_svc.best_score_)

# Update svc_model with the best estimator from GridSearchCV
svc_model = gs_svc.best_estimator_

Fitting 10 folds for each of 30 candidates, totalling 300 fits
Best params for LinearSVC: {'C': 100, 'class_weight': None, 'max_iter': 100}
Best CV f1 score for LinearSVC: 0.8271314658165755


### `SGDClassifier`

In [7]:
# Tune SGDClassifier with GridSearchCV

# Define parameter grid for SGDClassifier
param_grid_sgd = {
    'loss': ['hinge', 'log_loss', 'modified_huber', 'perceptron'],
    'alpha': [0.0001, 0.001, 0.01, 0.1],
    'class_weight': [None, 'balanced'],
    'max_iter': [100, 500, 1000]
}

sgd_base = SGDClassifier(random_state=42)
gs_sgd = GridSearchCV(sgd_base, param_grid_sgd, cv=NUM_FOLD, scoring='f1', n_jobs=-1, verbose=2)
gs_sgd.fit(X, y)

print("Best params for SGDClassifier:", gs_sgd.best_params_)
print("Best CV f1 score for SGDClassifier:", gs_sgd.best_score_)

# Update sgd_model with the best estimator from GridSearchCV
sgd_model = gs_sgd.best_estimator_

Fitting 10 folds for each of 96 candidates, totalling 960 fits
Best params for SGDClassifier: {'alpha': 0.0001, 'class_weight': None, 'loss': 'hinge', 'max_iter': 100}
Best CV f1 score for SGDClassifier: 0.8263128594910247


### `LogisticRegression`

In [8]:
# Tune a base LogisticRegression with GridSearchCV (independent from meta-model)

# Define parameter grid for LogisticRegression
param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100],
    'class_weight': [None, 'balanced'],
    'solver': ['liblinear', 'saga'],
    'max_iter': [100, 200, 500]
}

lr_base = LogisticRegression(random_state=42)
gs_lr = GridSearchCV(lr_base, param_grid_lr, cv=NUM_FOLD, scoring='f1', n_jobs=-1, verbose=2)
gs_lr.fit(X, y)

print("Best params for LogisticRegression:", gs_lr.best_params_)
print("Best CV f1 score for LogisticRegression:", gs_lr.best_score_)

# Update lr_model with the best estimator from GridSearchCV
lr_model = gs_lr.best_estimator_

Fitting 10 folds for each of 60 candidates, totalling 600 fits
Best params for LogisticRegression: {'C': 10, 'class_weight': None, 'max_iter': 100, 'solver': 'liblinear'}
Best CV f1 score for LogisticRegression: 0.8270403869909868


### `PassiveAggressiveClassifier`

In [9]:
# Tune PassiveAggressiveClassifier with GridSearchCV

# Define parameter grid for PassiveAggressiveClassifier
param_grid_pa = {
    'C': [0.01, 0.1, 1, 10, 100],
    'class_weight': [None, 'balanced'],
    'max_iter': [100, 500, 1000],
    'loss': ['hinge', 'squared_hinge'],
    'early_stopping': [True]
}

pa_base = PassiveAggressiveClassifier(random_state=42)
gs_pa = GridSearchCV(pa_base, param_grid_pa, cv=NUM_FOLD, scoring='f1', n_jobs=-1, verbose=2)
gs_pa.fit(X, y)

print("Best params for PassiveAggressiveClassifier:", gs_pa.best_params_)
print("Best CV f1 score for PassiveAggressiveClassifier:", gs_pa.best_score_)

# Update pa_model with the best estimator from GridSearchCV
pa_model = gs_pa.best_estimator_

Fitting 10 folds for each of 60 candidates, totalling 600 fits
Best params for PassiveAggressiveClassifier: {'C': 0.01, 'class_weight': None, 'early_stopping': True, 'loss': 'hinge', 'max_iter': 100}
Best CV f1 score for PassiveAggressiveClassifier: 0.8157997315228671


### `GaussianNB`

In [10]:
# Tune GaussianNB with GridSearchCV

# Define parameter grid for GaussianNB
param_grid_gnb = {
    'var_smoothing': [1e-09, 1e-08, 1e-07, 1e-06, 1e-05]
}

gnb_base = GaussianNB()
gs_gnb = GridSearchCV(gnb_base, param_grid_gnb, cv=NUM_FOLD, scoring='f1', n_jobs=-1, verbose=2)
gs_gnb.fit(X, y)

print("Best params for GaussianNB:", gs_gnb.best_params_)
print("Best CV f1 score for GaussianNB:", gs_gnb.best_score_)

# Update gnb_model with the best estimator from GridSearchCV
gnb_model = gs_gnb.best_estimator_

Fitting 10 folds for each of 5 candidates, totalling 50 fits
Best params for GaussianNB: {'var_smoothing': 1e-09}
Best CV f1 score for GaussianNB: 0.6565652648443525


### `ExtraTreesClassifier`

In [11]:
# Tune ExtraTreesClassifier with GridSearchCV

# Define parameter grid for ExtraTreesClassifier
param_grid_etc = {
    'n_estimators': [10, 50, 100],
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 3, 5],
    'min_samples_split': [2, 5],
    'class_weight': [None, 'balanced']
}

etc_base = ExtraTreesClassifier(random_state=42)
gs_etc = GridSearchCV(etc_base, param_grid_etc, cv=NUM_FOLD, scoring='f1', n_jobs=-1, verbose=2)
gs_etc.fit(X, y)

print("Best params for ExtraTreesClassifier:", gs_etc.best_params_)
print("Best CV f1 score for ExtraTreesClassifier:", gs_etc.best_score_)

# Update etc_model with the best estimator from GridSearchCV
etc_model = gs_etc.best_estimator_

Fitting 10 folds for each of 72 candidates, totalling 720 fits
Best params for ExtraTreesClassifier: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}
Best CV f1 score for ExtraTreesClassifier: 0.8269366921895769


### `AdaBoostClassifier`

In [14]:
# Tune AdaBoostClassifier with GridSearchCV

# Define parameter grid for AdaBoostClassifier
param_grid_abc = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 1.0],
}

adb_base = AdaBoostClassifier(random_state=42)
gs_adb = GridSearchCV(adb_base, param_grid_abc, cv=NUM_FOLD, scoring='f1', n_jobs=-1, verbose=2)
gs_adb.fit(X, y)

print("Best params for AdaBoostClassifier:", gs_adb.best_params_)
print("Best CV f1 score for AdaBoostClassifier:", gs_adb.best_score_)

# Update adb_model with the best estimator from GridSearchCV
adb_model = gs_adb.best_estimator_

Fitting 10 folds for each of 9 candidates, totalling 90 fits
Best params for AdaBoostClassifier: {'learning_rate': 1.0, 'n_estimators': 200}
Best CV f1 score for AdaBoostClassifier: 0.8286866754310452


## K-Fold Cross Validation Training

In [17]:
# Base models for stacking ensemble
base_models = [dt_model, svc_model, sgd_model, lr_model, pa_model, gnb_model, etc_model, adb_model]

# K-fold Cross Validation Training
kf = KFold(n_splits=NUM_FOLD, shuffle=True, random_state=42)

fold = 1 # Fold counter
classification_reports = [] # To store classification reports for each fold

for train_index, val_index in kf.split(X):
    print(f"Training fold {fold}...")
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    
    # Define Stacking Classifier
    stack_clf = StackingClassifier(classifiers=base_models, meta_classifier=META_MODEL)
    
    # Train the stacking classifier
    stack_clf.fit(X_train, y_train)
    
    # Evaluate on validation set
    y_pred = stack_clf.predict(X_val)
    cr = classification_report(y_val, y_pred, output_dict=True)
    cm = confusion_matrix(y_val, y_pred)
    print(f"Fold {fold} results:")
    print(cr)
    print("Confusion Matrix:")
    print(cm)
    
    # Store report
    classification_reports.append(cr)
    
    print("\n")
    fold += 1

Training fold 1...
Fold 1 results:
{'0': {'precision': 0.9667782869228058, 'recall': 0.9530055594162613, 'f1-score': 0.9598425196850394, 'support': 11512.0}, '1': {'precision': 0.8012490815576782, 'recall': 0.8526192337763878, 'f1-score': 0.8261363636363637, 'support': 2558.0}, 'accuracy': 0.9347547974413646, 'macro avg': {'precision': 0.884013684240242, 'recall': 0.9028123965963246, 'f1-score': 0.8929894416607016, 'support': 14070.0}, 'weighted avg': {'precision': 0.9366842068002758, 'recall': 0.9347547974413646, 'f1-score': 0.9355340372989334, 'support': 14070.0}}
Confusion Matrix:
[[10971   541]
 [  377  2181]]


Training fold 2...
Fold 2 results:
{'0': {'precision': 0.9665696392343653, 'recall': 0.9550287606763117, 'f1-score': 0.9607645434220332, 'support': 11474.0}, '1': {'precision': 0.8111964873765093, 'recall': 0.8540061633281972, 'f1-score': 0.8320510414711954, 'support': 2596.0}, 'accuracy': 0.9363894811656006, 'macro avg': {'precision': 0.8888830633054373, 'recall': 0.904517

## Cross Validation Results

In [18]:
avg_report = {}

# Extract average metrics across folds
for key in classification_reports[0].keys():
    if key in ['0', '1', 'macro avg', 'weighted avg']: # Average the metric sections
        avg_report[key] = {}
        for metric in classification_reports[0][key].keys(): # Iterate through metrics ('precision', 'recall', etc.)
            avg_report[key][metric] = np.mean([report[key][metric] for report in classification_reports])
    elif key == 'accuracy':
        avg_report[key] = np.mean([report[key] for report in classification_reports])

# Format and print the average classification report
headers = ["precision", "recall", "f1-score", "support"]
table = []
for label in ['0', '1', 'macro avg', 'weighted avg']:
    if label in avg_report:
        row = [
            label,
            f"{avg_report[label]['precision']:.4f}",
            f"{avg_report[label]['recall']:.4f}",
            f"{avg_report[label]['f1-score']:.4f}",
            f"{int(avg_report[label]['support']):d}"
        ]
        table.append(row)

print(tabulate(table, headers=headers, floatfmt=".4f", numalign="right"))
print(f"\nAverage Accuracy: {avg_report['accuracy']:.4f}")

                precision    recall    f1-score    support
------------  -----------  --------  ----------  ---------
0                  0.9662    0.9543      0.9602      11513
1                  0.8049    0.8495      0.8266       2556
macro avg          0.8855    0.9019      0.8934      14070
weighted avg       0.9369    0.9352      0.9359      14070

Average Accuracy: 0.9352


## Train Final Model on Full Dataset

In [19]:
# Train Final Model on Full Dataset
final_model = StackingClassifier(classifiers=base_models, meta_classifier=META_MODEL)

final_model.fit(X, y)

,classifiers,"[DecisionTreeC...ndom_state=42), LinearSVC(C=1...ndom_state=42), ...]"
,meta_classifier,LogisticRegre...r='liblinear')
,use_probas,False
,drop_proba_col,None
,average_probas,False
,verbose,0
,use_features_in_secondary,False
,store_train_meta_features,False
,use_clones,True
,fit_base_estimators,True
,penalty,'l2'


In [20]:
# Predict on test data
test_preds = final_model.predict(test_data)

# Prepare submission dataframe
submission_df = pd.DataFrame({
    "id": test_ids,
    "Depression": test_preds
})

# Save submission file
submission_df.to_csv("../data/HCMUS-22KHDL-Cloud_Djata_stack_v5.csv", index=False)